### PACOTES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import itertools

from sklearn.preprocessing import MinMaxScaler

import plotly.express as px 
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ks_2samp
from sklearn.preprocessing import RobustScaler
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (

    silhouette_score,
    davies_bouldin_score

)
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import mahalanobis
import time





In [2]:

inicio = time.time()
df_resumo = pd.read_csv('resumo_features.csv')
df = pd.read_csv('creditcard.csv')

# AJUSTE DE TIPOS
df_resumo['Comparacao_dis_whith'] = pd.to_numeric(
    df_resumo['Comparacao_dis_whith'],
    errors='coerce'
)

# MANN-WHITNEY SCORE
# Quanto menor o p-value -> melhor
df_resumo['MannWhitney_Score'] = -np.log10(
    df_resumo['Comparacao_dis_whith'] + 1e-300
)

# CORRELAÇÃO ABSOLUTA
df_resumo['Corr_spearman_abs'] = abs(
    df_resumo['Correlacao_spearman']
)

# AJUSTE DE MÉTRICAS DISTRIBUCIONAIS

# KL e Wasser podem explodir
# Aplicamos log para estabilizar

df_resumo['divergencia_kl'] = np.log1p(
    abs(df_resumo['divergencia_kl'])
)

df_resumo['Wasser'] = np.log1p(
    abs(df_resumo['Wasser'])
)

# AJUSTE DE NORMALIDADE

# Quanto MENOR distância da normalidade -> melhor
df_resumo['Normalidade_Score'] = 1 / (
    1 + abs(df_resumo['Normalidade'])
)

# AJUSTE DE ASSIMETRIA
df_resumo['Skewness_Score'] = 1 / (
    1 + abs(df_resumo['Assimetria'])
)


# AJUSTE DE CURTOSE
df_resumo['Kurtosis_Score'] = 1 / (
    1 + abs(df_resumo['Curtose'])
)

# AJUSTE DE OUTLIERS
df_resumo['Outlier_Score'] = 1 / (
    1 + df_resumo['Qnd_Outliers']
)

# MÉTRICAS UTILIZADAS
metricas = [

    # INDISPENSÁVEL
    'Curva ROC',
    'KS',

    # MUITO IMPORTANTE
    'divergencia_kl',
    'Wasser',

    # IMPORTANTE
    'inf_mutua',
    'MannWhitney_Score',

    # BOM
    'Normalidade_Score',
    'Sep_mediana_norm',

    # ÚTIL
    'Skewness_Score',
    'Kurtosis_Score',

    # ACESSÓRIO
    'Corr_spearman_abs',
    'Outlier_Score'
]


# TRATAMENTO DE NaN
df_resumo[metricas] = df_resumo[
    metricas
].fillna(0)


# NORMALIZAÇÃO ROBUSTA
# Melhor para fraude/outliers

scaler = RobustScaler()

df_norm = df_resumo.copy()

df_norm[metricas] = scaler.fit_transform(
    df_norm[metricas]
)


# MIN-MAX FINAL
# Após robust scaling

for col in metricas:

    minimo = df_norm[col].min()
    maximo = df_norm[col].max()

    df_norm[col] = (
        (df_norm[col] - minimo)
        /
        (maximo - minimo + 1e-9)
    )

# =========================================================
# SCORE FINAL
# =========================================================

df_norm['Score_Final'] = (

    # INDISPENSÁVEL -> 0.25
    df_norm['Curva ROC'] * 0.125 +
    df_norm['KS'] * 0.125 +

    # MUITO IMPORTANTE -> 0.21
    df_norm['divergencia_kl'] * 0.105 +
    df_norm['Wasser'] * 0.105 +

    # IMPORTANTE -> 0.17
    df_norm['inf_mutua'] * 0.085 +
    df_norm['MannWhitney_Score'] * 0.085 +

    # BOM -> 0.13
    df_norm['Normalidade_Score'] * 0.065 +
    df_norm['Sep_mediana_norm'] * 0.065 +

    # ÚTIL -> 0.09
    df_norm['Skewness_Score'] * 0.045 +
    df_norm['Kurtosis_Score'] * 0.045 +

    # ACESSÓRIO -> 0.05
    df_norm['Corr_spearman_abs'] * 0.025 +
    df_norm['Outlier_Score'] * 0.025
)


# GARANTIR SEM NaN
df_norm['Score_Final'] = df_norm[
    'Score_Final'
].fillna(0)

# RANKING FINAL
ranking = df_norm.sort_values(
    'Score_Final',
    ascending=False
).reset_index(drop=True)

# POSIÇÃO
ranking.insert(
    0,
    'Posicao_Rank',
    ranking.index + 1
)

# EXIBIÇÃO
pd.set_option(
    'display.max_columns',
    None
)

display(ranking)

# SALVAR CSV
ranking.to_csv(
    'individual_score_feature_individuais.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

,Posicao_Rank,Feature,Normalidade,Qnd_Outliers,Sep_mediana_norm,Comparacao_dis_whith,Correlacao_spearman,inf_mutua,divergencia_kl,Curva ROC,KS,Wasser,Assimetria,Curtose,MannWhitney_Score,Corr_spearman_abs,Normalidade_Score,Skewness_Score,Kurtosis_Score,Outlier_Score,Score_Final
0,1,V14,0,14149,1.000000,1.471581e-260,-0.064613,0.984641,1.000000,1.000000,1.000000,0.211410,1.995165,23.879022,1.000000,1.000000,0.0,0.305756,0.046824,0.000045,0.651760
1,2,V12,0,15348,0.796989,8.416027e-247,-0.062870,0.917286,0.840947,0.972118,0.924927,0.201544,2.278389,20.241493,0.946936,0.972125,0.0,0.274060,0.055062,0.000040,0.595970
2,3,V10,0,9496,0.579452,9.611131e-222,-0.059564,0.908347,0.894367,0.919245,0.950893,0.192730,1.187134,31.987656,0.850284,0.919255,0.0,0.441298,0.035000,0.000080,0.579477
3,4,V11,0,780,0.497847,4.910592e-226,0.060143,0.820345,0.725013,0.928507,0.889460,0.157954,0.356504,1.633872,0.866838,0.928514,0.0,0.748947,0.453109,0.001255,0.573065
4,5,V4,0,11148,0.415041,3.625904e-248,0.063045,0.586806,0.805397,0.974920,0.902628,0.173101,0.676289,2.635388,0.952203,0.974924,0.0,0.594410,0.327929,0.000064,0.561109
5,6,V17,0,7420,0.870823,9.219384e-124,-0.044335,1.000000,0.905760,0.675706,0.875582,0.212480,3.844894,94.798034,0.472352,0.675708,0.0,0.165685,0.011212,0.000109,0.517935
6,7,V3,0,3363,0.486416,1.211048e-219,-0.059278,0.583784,0.718940,0.914680,0.822726,0.212220,2.240144,26.619062,0.842183,0.914681,0.0,0.278017,0.042052,0.000272,0.505049
7,8,V16,0,8184,0.580474,1.808172e-156,-0.049936,0.733854,0.745415,0.765280,0.800845,0.165757,1.100960,10.418927,0.598510,0.765281,0.0,0.461906,0.103528,0.000097,0.486999
8,9,V7,0,8948,0.346706,1.464234e-146,-0.048308,0.457887,0.709440,0.739240,0.767815,0.191285,2.553894,405.600275,0.560292,0.739245,0.0,0.248076,0.001663,0.000086,0.421761
9,10,V9,0,8283,0.272512,8.943723e-154,-0.049499,0.498804,0.482208,0.758284,0.660477,0.127076,0.554677,3.731224,0.588117,0.758292,0.0,0.645688,0.251678,0.000095,0.410763



CSV salvo com sucesso!


### TOP 10


In [3]:

# CARREGAR CSV DO RANKING
ranking = pd.read_csv('individual_score_feature_individuais.csv')

ranking = ranking.sort_values(
    by='Score_Final',
    ascending=False
)

top10 = ranking.head(10)

# ATRIBUINDO ÀS VARIÁVEIS
Primeiro_lugar = top10.iloc[0]['Feature']
Segundo_lugar = top10.iloc[1]['Feature']
Terceiro_lugar = top10.iloc[2]['Feature']
Quarto_lugar = top10.iloc[3]['Feature']
Quinto_lugar = top10.iloc[4]['Feature']
Sexto_lugar = top10.iloc[5]['Feature']
Setimo_lugar = top10.iloc[6]['Feature']
Oitavo_lugar = top10.iloc[7]['Feature']
Nono_lugar = top10.iloc[8]['Feature']
Decimo_lugar = top10.iloc[9]['Feature']

# PRINT TOP 10
print('TOP 10 FEATURES:\n')

for i, row in top10.iterrows():

    print(
        f"{row['Posicao_Rank']}º -> "
        f"{row['Feature']}"
    )

# LISTA FINAL
features_top10 = top10['Feature'].tolist()
print('\nLista Top 10:\n')
print(features_top10)

TOP 10 FEATURES:

1º -> V14
2º -> V12
3º -> V10
4º -> V11
5º -> V4
6º -> V17
7º -> V3
8º -> V16
9º -> V7
10º -> V9

Lista Top 10:

['V14', 'V12', 'V10', 'V11', 'V4', 'V17', 'V3', 'V16', 'V7', 'V9']


 ### COMBINACOES 2X2 

In [4]:

y = df['status_fraude']

sample_size_silhouette = 14240

# PEGANDO TOP 10 FEATURES
features_top10 = top10['Feature'].tolist()


# COMBINAÇÕES 2x2
combinacoes_2x2 = list(
    itertools.combinations(
        features_top10,
        2
    )
)

print(
    f'Total de combinações: '
    f'{len(combinacoes_2x2)}\n'
)

# DICIONÁRIO
dic_combinacoes = {}

# LOOP PRINCIPAL

for i, (f1, f2) in enumerate(
    combinacoes_2x2,
    start=1
):

    try:

        # MATRIZ COMPLETA

        X = df[[f1, f2]].values

        # PADRONIZAÇÃO
        scaler = StandardScaler()

        X_scaled = scaler.fit_transform(X)

        # GMM

        gmm = GaussianMixture(

            n_components=2,
            covariance_type='diag',
            random_state=42,
            max_iter=100

        )

        gmm.fit(X_scaled)

        # LABELS

        labels = gmm.predict(X_scaled)

        # 1) SILHOUETTE
        # Apenas em 5% da base

        sample_size = min(
            sample_size_silhouette,
            len(X_scaled)
        )

        idx = np.random.choice(

            len(X_scaled),
            size=sample_size,
            replace=False

        )

        silhouette = silhouette_score(

            X_scaled[idx],
            labels[idx]

        )

        # 2) DAVIES-BOULDIN

        davies = davies_bouldin_score(
            X_scaled,
            labels
        )

        # 3) WASSERSTEIN

        classe_0 = X_scaled[y == 0]
        classe_1 = X_scaled[y == 1]

        wasser_1 = wasserstein_distance(

            classe_0[:, 0],
            classe_1[:, 0]

        )

        wasser_2 = wasserstein_distance(

            classe_0[:, 1],
            classe_1[:, 1]

        )

        wasser_final = (
            wasser_1 + wasser_2
        ) / 2

        # 4) MAHALANOBIS
        media_0 = np.mean(
            classe_0,
            axis=0
        )

        media_1 = np.mean(
            classe_1,
            axis=0
        )

        cov = np.cov(
            X_scaled.T
        )

        inv_cov = np.linalg.pinv(
            cov
        )

        maha = mahalanobis(

            media_0,
            media_1,
            inv_cov

        )

        # 5) OVERLAP GAUSSIANO
        probs = gmm.predict_proba(
            X_scaled
        )

        overlap = np.mean(
            np.min(probs, axis=1)
        )

        overlap_score = 1 - overlap

        # SALVANDO
        chave = f'Combo_{i}'

        dic_combinacoes[chave] = {

            'Feature_1': f1,
            'Feature_2': f2,

            'Silhouette': round(
                float(silhouette), 6
            ),

            'Davies_Bouldin': round(
                float(davies), 6
            ),

            'Wasserstein': round(
                float(wasser_final), 6
            ),

            'Mahalanobis': round(
                float(maha), 6
            ),

            'Overlap_Gaussiano': round(
                float(overlap_score), 6
            )
        }


        # PRINT

        print(

            f'{chave} -> '

            f'{f1} / {f2} '

            f'| SIL: {silhouette:.4f} '

            f'| DB: {davies:.4f} '

            f'| WASSER: {wasser_final:.4f} '

            f'| MAHA: {maha:.4f} '

            f'| OVERLAP: {overlap_score:.4f}'
        )

    except Exception as e:

        print(
            f'Erro em {f1} / {f2}: {e}'
        )

# DATAFRAME FINAL

df_combinacoes = pd.DataFrame.from_dict(
    dic_combinacoes,
    orient='index'
).reset_index()

df_combinacoes.rename(
    columns={'index': 'Combo'},
    inplace=True
)

display(df_combinacoes)

# SALVAR CSV
df_combinacoes.to_csv(
    '2x2_visu_metfeat.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

Total de combinações: 45

Combo_1 -> V14 / V12 | SIL: 0.5449 | DB: 1.7119 | WASSER: 6.7805 | MAHA: 9.6154 | OVERLAP: 0.9403
Combo_2 -> V14 / V10 | SIL: 0.6344 | DB: 2.5124 | WASSER: 6.2542 | MAHA: 8.9640 | OVERLAP: 0.9718
Combo_3 -> V14 / V11 | SIL: 0.4052 | DB: 4.4043 | WASSER: 5.5076 | MAHA: 8.1845 | OVERLAP: 0.8536
Combo_4 -> V14 / V4 | SIL: 0.4235 | DB: 3.1432 | WASSER: 5.2496 | MAHA: 7.9627 | OVERLAP: 0.8497
Combo_5 -> V14 / V17 | SIL: 0.6102 | DB: 1.6574 | WASSER: 7.8023 | MAHA: 10.7185 | OVERLAP: 0.9585
Combo_6 -> V14 / V3 | SIL: 0.5269 | DB: 2.0791 | WASSER: 5.9662 | MAHA: 8.6411 | OVERLAP: 0.9214
Combo_7 -> V14 / V16 | SIL: 0.4595 | DB: 4.8672 | WASSER: 6.0259 | MAHA: 8.6877 | OVERLAP: 0.8718
Combo_8 -> V14 / V7 | SIL: 0.6535 | DB: 4.7946 | WASSER: 5.9044 | MAHA: 8.5680 | OVERLAP: 0.9714
Combo_9 -> V14 / V9 | SIL: 0.4266 | DB: 2.8120 | WASSER: 4.8196 | MAHA: 7.6561 | OVERLAP: 0.8593
Combo_10 -> V12 / V10 | SIL: 0.4696 | DB: 1.5714 | WASSER: 5.7490 | MAHA: 8.1642 | OVERLAP: 0.8

,Combo,Feature_1,Feature_2,Silhouette,Davies_Bouldin,Wasserstein,Mahalanobis,Overlap_Gaussiano
0,Combo_1,V14,V12,0.544928,1.711897,6.780460,9.615413,0.940298
1,Combo_2,V14,V10,0.634397,2.512369,6.254177,8.964034,0.971761
2,Combo_3,V14,V11,0.405153,4.404303,5.507589,8.184543,0.853600
3,Combo_4,V14,V4,0.423455,3.143184,5.249637,7.962672,0.849717
4,Combo_5,V14,V17,0.610185,1.657416,7.802251,10.718511,0.958545
5,Combo_6,V14,V3,0.526946,2.079137,5.966202,8.641100,0.921435
6,Combo_7,V14,V16,0.459526,4.867179,6.025882,8.687736,0.871774
7,Combo_8,V14,V7,0.653481,4.794604,5.904431,8.568019,0.971352
8,Combo_9,V14,V9,0.426596,2.812050,4.819570,7.656133,0.859266
9,Combo_10,V12,V10,0.469602,1.571416,5.748965,8.164246,0.895896



CSV salvo com sucesso!


### COMBINACAO 3X3

In [5]:

# TARGET

y = df['status_fraude']

# AMOSTRA PARA SILHOUETTE
sample_size_silhouette = 14240
# TOP FEATURES
features_top10 = top10['Feature'].tolist()

# COMBINAÇÕES 3x3
combinacoes_3x3 = list(
    itertools.combinations(features_top10, 3)
)

print(f'Total de combinações: {len(combinacoes_3x3)}\n')

# DICIONÁRIO
dic_combinacoes = {}
# LOOP PRINCIPAL
for i, (f1, f2, f3) in enumerate(combinacoes_3x3, start=1):

    try:
        # MATRIZ
        X = df[[f1, f2, f3]].values

        # PADRONIZAÇÃO
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)


        # GMM
        gmm = GaussianMixture(
            n_components=2,
            covariance_type='diag',
            random_state=42,
            max_iter=100
        )

        gmm.fit(X_scaled)

        labels = gmm.predict(X_scaled)

        # SILHOUETTE (5%)
        sample_size = min(sample_size_silhouette, len(X_scaled))

        idx = np.random.choice(
            len(X_scaled),
            size=sample_size,
            replace=False
        )

        silhouette = silhouette_score(
            X_scaled[idx],
            labels[idx]
        )

      
        # DAVIES-BOULDIN
        davies = davies_bouldin_score(X_scaled, labels)

        # WASSERSTEIN (3 features)
        classe_0 = X_scaled[y == 0]
        classe_1 = X_scaled[y == 1]

        wasser_1 = wasserstein_distance(classe_0[:, 0], classe_1[:, 0])
        wasser_2 = wasserstein_distance(classe_0[:, 1], classe_1[:, 1])
        wasser_3 = wasserstein_distance(classe_0[:, 2], classe_1[:, 2])

        wasser_final = (wasser_1 + wasser_2 + wasser_3) / 3

        # MAHALANOBIS
        media_0 = np.mean(classe_0, axis=0)
        media_1 = np.mean(classe_1, axis=0)

        cov = np.cov(X_scaled.T)
        inv_cov = np.linalg.pinv(cov)

        maha = mahalanobis(media_0, media_1, inv_cov)

        # OVERLAP GAUSSIANO
        probs = gmm.predict_proba(X_scaled)

        overlap = np.mean(np.min(probs, axis=1))
        overlap_score = 1 - overlap

        # SALVAR
        chave = f'Combo3D_{i}'

        dic_combinacoes[chave] = {

            'Feature_1': f1,
            'Feature_2': f2,
            'Feature_3': f3,

            'Silhouette': round(float(silhouette), 6),
            'Davies_Bouldin': round(float(davies), 6),
            'Wasserstein': round(float(wasser_final), 6),
            'Mahalanobis': round(float(maha), 6),
            'Overlap_Gaussiano': round(float(overlap_score), 6)

        }

        # PRINT
        print(
            f'{chave} -> {f1} / {f2} / {f3} '
            f'| SIL: {silhouette:.4f} '
            f'| DB: {davies:.4f} '
            f'| WASSER: {wasser_final:.4f} '
            f'| MAHA: {maha:.4f} '
            f'| OVERLAP: {overlap_score:.4f}'
        )

    except Exception as e:
        print(f'Erro em {f1} / {f2} / {f3}: {e}')

# DATAFRAME FINAL
df_combinacoes = pd.DataFrame.from_dict(
    dic_combinacoes,
    orient='index'
).reset_index()

df_combinacoes.rename(
    columns={'index': 'Combo'},
    inplace=True
)

# EXIBIR

display(df_combinacoes)

# SALVAR CSV
df_combinacoes.to_csv(
    '3x3_visu_metfeat.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

fim = time.time()
print(f"\nTempo total de execução: {fim - inicio:.2f} segundos")

Total de combinações: 120



Combo3D_1 -> V14 / V12 / V10 | SIL: 0.4621 | DB: 2.5252 | WASSER: 6.2612 | MAHA: 10.9422 | OVERLAP: 0.9383
Combo3D_2 -> V14 / V12 / V11 | SIL: 0.4496 | DB: 2.1032 | WASSER: 5.7635 | MAHA: 10.3134 | OVERLAP: 0.9362
Combo3D_3 -> V14 / V12 / V4 | SIL: 0.4114 | DB: 2.4455 | WASSER: 5.5915 | MAHA: 10.1382 | OVERLAP: 0.9116
Combo3D_4 -> V14 / V12 / V17 | SIL: 0.4697 | DB: 1.9935 | WASSER: 7.2932 | MAHA: 12.4203 | OVERLAP: 0.9437
Combo3D_5 -> V14 / V12 / V3 | SIL: 0.4382 | DB: 2.4021 | WASSER: 6.0692 | MAHA: 10.6793 | OVERLAP: 0.9293
Combo3D_6 -> V14 / V12 / V16 | SIL: 0.4110 | DB: 2.5010 | WASSER: 6.1090 | MAHA: 10.7171 | OVERLAP: 0.9183
Combo3D_7 -> V14 / V12 / V7 | SIL: 0.4911 | DB: 2.5347 | WASSER: 6.0280 | MAHA: 10.6202 | OVERLAP: 0.9556
Combo3D_8 -> V14 / V12 / V9 | SIL: 0.4574 | DB: 2.1704 | WASSER: 5.3048 | MAHA: 9.8992 | OVERLAP: 0.9349
Combo3D_9 -> V14 / V10 / V11 | SIL: 0.5577 | DB: 2.8751 | WASSER: 5.4126 | MAHA: 9.7089 | OVERLAP: 0.9743
Combo3D_10 -> V14 / V10 / V4 | SIL: 0.5166 

,Combo,Feature_1,Feature_2,Feature_3,Silhouette,Davies_Bouldin,Wasserstein,Mahalanobis,Overlap_Gaussiano
0,Combo3D_1,V14,V12,V10,0.462069,2.525182,6.261201,10.942234,0.938298
1,Combo3D_2,V14,V12,V11,0.449584,2.103152,5.763475,10.313357,0.936230
2,Combo3D_3,V14,V12,V4,0.411392,2.445530,5.591507,10.138182,0.911582
3,Combo3D_4,V14,V12,V17,0.469655,1.993484,7.293250,12.420349,0.943746
4,Combo3D_5,V14,V12,V3,0.438152,2.402055,6.069217,10.679288,0.929349
...,...,...,...,...,...,...,...,...,...
115,Combo3D_116,V17,V7,V9,0.608653,6.446502,5.065162,9.363815,0.982240
116,Combo3D_117,V3,V16,V7,0.600253,3.142120,4.645338,8.020197,0.983733
117,Combo3D_118,V3,V16,V9,0.533636,2.140000,3.922097,7.037684,0.970332
118,Combo3D_119,V3,V7,V9,0.599462,3.122408,3.841130,6.889352,0.987117



CSV salvo com sucesso!

Tempo total de execução: 1050.71 segundos
